In [1]:
pip install polars

In [2]:
import polars as pl


schema_overrides = {
    "age": pl.Float64
}

alzheimers_df = pl.read_csv(
    "\alzheimers_balanced.csv",
    schema_overrides=schema_overrides
)

print("Balanced dataset shape:", alzheimers_df.shape)
print(alzheimers_df.head())
print("✅ Done!")



# Collect the sample IDs
alzheimers_samples = alzheimers_df["sample_id"].to_list()
print("Number of samples:", len(alzheimers_samples))

# Subset from the huge train.csv with only these columns
use_cols = ["cpgsite"] + alzheimers_samples
train_data = pl.read_csv("ai4bio_trainset\traindata.csv", has_header=True, columns=use_cols)

print("Subset methylation shape:", train_data.shape)

# Save as CSV (ready for ML)
train_data.write_csv("alzheimers_methylation.csv")
print("✅ Saved subset as alzheimers_methylation.csv")


Balanced dataset shape: (1000, 6)
shape: (5, 6)
┌──────┬────────────┬──────┬────────┬────────────────┬─────────────────────┐
│ s_no ┆ sample_id  ┆ age  ┆ gender ┆ sample_type    ┆ disease             │
│ ---  ┆ ---        ┆ ---  ┆ ---    ┆ ---            ┆ ---                 │
│ i64  ┆ str        ┆ f64  ┆ str    ┆ str            ┆ str                 │
╞══════╪════════════╪══════╪════════╪════════════════╪═════════════════════╡
│ 1    ┆ train10001 ┆ 88.0 ┆ F      ┆ disease tissue ┆ Alzheimer's disease │
│ 3    ┆ train10003 ┆ 93.0 ┆ F      ┆ disease tissue ┆ Alzheimer's disease │
│ 4    ┆ train10004 ┆ 96.0 ┆ F      ┆ disease tissue ┆ Alzheimer's disease │
│ 6    ┆ train10006 ┆ 80.0 ┆ M      ┆ disease tissue ┆ Alzheimer's disease │
│ 7    ┆ train10007 ┆ 79.0 ┆ F      ┆ disease tissue ┆ Alzheimer's disease │
└──────┴────────────┴──────┴────────┴────────────────┴─────────────────────┘
✅ Done!
Number of samples: 1000
Subset methylation shape: (485512, 1001)
✅ Saved subset as alzheimers_met

In [7]:
import polars as pl
import os

file_path = "alzheimers_methylation.csv"

# Step 1: Check if file exists
if os.path.exists(file_path):
    print(f"✅ File found: {file_path}")
    print(f"📦 File size: {os.path.getsize(file_path) / (1024*1024):.2f} MB")

    try:
        # Step 2: Try reading the first 5 rows only
        df_head = pl.read_csv(file_path, n_rows=5)
        print("\n✅ Header and first 5 rows loaded successfully:")
        print(df_head)

        # Step 3: Load fully (if small enough)
        df = pl.read_csv(file_path)
        print(f"\n✅ Full dataset loaded successfully with shape: {df.shape}")

        # Step 4: Extra check – column names and null counts
        print("\n📋 Columns:", df.columns)
        print("\n🔎 Null values per column:")
        print(df.null_count())

    except Exception as e:
        print("❌ Error while reading file:", e)

else:
    print(f"❌ File not found at: {file_path}")


✅ File found: alzheimers_methylation.csv
📦 File size: 7024.99 MB

✅ Header and first 5 rows loaded successfully:
shape: (5, 1_001)
┌────────────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ cpgsite    ┆ train10001 ┆ train10003 ┆ train10004 ┆ … ┆ train16962 ┆ train16970 ┆ train16997 ┆ train16999 │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│ str        ┆ f64        ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64        ┆ f64        ┆ f64        │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ cg00050873 ┆ null       ┆ null       ┆ null       ┆ … ┆ 1.790011   ┆ 2.163468   ┆ 1.621      ┆ null       │
│ cg00212031 ┆ null       ┆ null       ┆ null       ┆ … ┆ -4.80796   ┆ -3.580953  ┆ -3.886935  ┆ null       │
│ cg00213748 ┆ null       ┆ null       ┆ null       ┆ … ┆ null       ┆ 2.375206   ┆ null       ┆ nu

In [5]:
!pip install pandas
!pip install pyarrow


In [7]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

# File paths
traindata_file = "ai4bio_trainset\traindata.csv"
alz_balance_file = "alzheimers_balanced.csv"
output_dir = "alz_1000samples_parquet_chunks"

# Create output directory if not exists
os.makedirs(output_dir, exist_ok=True)

# 1️⃣ Load 1000 Alzheimer sample IDs
alz_df = pd.read_csv(alz_balance_file)
alz_samples = list(alz_df['sample_id'])
usecols = ['cpgsite'] + alz_samples  # Only needed columns

print(f"Extracting {len(alz_samples)} Alzheimer samples from methylation data...")

# 2️⃣ Stream methylation data in chunks and write separate Parquet files
chunksize = 10000  # number of CpGs per chunk

for i, chunk in enumerate(pd.read_csv(traindata_file, usecols=usecols, chunksize=chunksize)):
    # chunk: rows = CpGs, columns = samples + 'cpgsite'
    cpg_ids = chunk['cpgsite'].tolist()
    data = chunk.drop('cpgsite', axis=1).T  # transpose: samples × CpGs
    data.columns = cpg_ids                 # CpG IDs as columns

    # Convert to Arrow Table
    table_chunk = pa.Table.from_pandas(data)

    # Write each chunk as separate Parquet file
    output_file = os.path.join(output_dir, f"alz_chunk_{i+1}.parquet")
    pq.write_table(table_chunk, output_file)

    print(f"✅ Chunk {i+1} saved ({len(cpg_ids)} CpGs)")

print(f"All chunks saved in directory '{output_dir}'")


Extracting 1000 Alzheimer samples from methylation data...
✅ Chunk 1 saved (10000 CpGs)
✅ Chunk 2 saved (10000 CpGs)
✅ Chunk 3 saved (10000 CpGs)
✅ Chunk 4 saved (10000 CpGs)
✅ Chunk 5 saved (10000 CpGs)
✅ Chunk 6 saved (10000 CpGs)
✅ Chunk 7 saved (10000 CpGs)
✅ Chunk 8 saved (10000 CpGs)
✅ Chunk 9 saved (10000 CpGs)
✅ Chunk 10 saved (10000 CpGs)
✅ Chunk 11 saved (10000 CpGs)
✅ Chunk 12 saved (10000 CpGs)
✅ Chunk 13 saved (10000 CpGs)
✅ Chunk 14 saved (10000 CpGs)
✅ Chunk 15 saved (10000 CpGs)
✅ Chunk 16 saved (10000 CpGs)
✅ Chunk 17 saved (10000 CpGs)
✅ Chunk 18 saved (10000 CpGs)
✅ Chunk 19 saved (10000 CpGs)
✅ Chunk 20 saved (10000 CpGs)
✅ Chunk 21 saved (10000 CpGs)
✅ Chunk 22 saved (10000 CpGs)
✅ Chunk 23 saved (10000 CpGs)
✅ Chunk 24 saved (10000 CpGs)
✅ Chunk 25 saved (10000 CpGs)
✅ Chunk 26 saved (10000 CpGs)
✅ Chunk 27 saved (10000 CpGs)
✅ Chunk 28 saved (10000 CpGs)
✅ Chunk 29 saved (10000 CpGs)
✅ Chunk 30 saved (10000 CpGs)
✅ Chunk 31 saved (10000 CpGs)
✅ Chunk 32 saved (10

In [8]:
import pyarrow.parquet as pq

# Load a single chunk
table = pq.read_table("alz_1000samples_parquet_chunks")

# Convert to pandas DataFrame
df = table.to_pandas()
print(df.shape)
print(df.head())


(49000, 10000)
            cg00050873  cg00212031  cg00213748  cg00214611  cg00455876  \
train10001         NaN         NaN         NaN         NaN         NaN   
train10003         NaN         NaN         NaN         NaN         NaN   
train10004         NaN         NaN         NaN         NaN         NaN   
train10006    1.423834    -9.21044    1.035353   -3.837361    1.288795   
train10007         NaN         NaN         NaN         NaN         NaN   

            cg01707559  cg02004872  cg02011394  cg02050847  cg02233190  ...  \
train10001         NaN         NaN         NaN         NaN         NaN  ...   
train10003         NaN         NaN         NaN         NaN         NaN  ...   
train10004   -1.398461         NaN         NaN         NaN         NaN  ...   
train10006   -3.100385   -5.093549    2.767818    1.848299   -3.580953  ...   
train10007         NaN         NaN         NaN         NaN         NaN  ...   

            cg14976971  cg14978304  cg14978994  cg14981312  cg149

In [9]:
# Load one chunk
import pyarrow.parquet as pq
table = pq.read_table("alz_1000samples_parquet_chunks")
df = table.to_pandas()

# Check all samples (rows)
print(df.index)  # or
print(df.shape)


Index(['train10001', 'train10003', 'train10004', 'train10006', 'train10007',
       'train10008', 'train10010', 'train10011', 'train10012', 'train10013',
       ...
       'train16878', 'train16892', 'train16914', 'train16933', 'train16955',
       'train16958', 'train16962', 'train16970', 'train16997', 'train16999'],
      dtype='object', length=49000)
(49000, 10000)


In [10]:
import pandas as pd
import pyarrow.parquet as pq
import glob
import os

parquet_dir = "alz_1000samples_parquet_chunks"
output_csv_dir = "alz_1000samples_csv_chunks"
os.makedirs(output_csv_dir, exist_ok=True)

# 1️⃣ List all Parquet chunk files
chunk_files = sorted(glob.glob(f"{parquet_dir}/*.parquet"))

for i, f in enumerate(chunk_files):
    table = pq.read_table(f)
    df = table.to_pandas()  # samples × CpGs

    # 2️⃣ Save as CSV
    csv_file = os.path.join(output_csv_dir, f"alz_chunk_{i+1}.csv")
    df.to_csv(csv_file, index=True)  # index = sample IDs

    print(f"✅ Saved chunk {i+1} as CSV: {csv_file}")


✅ Saved chunk 1 as CSV: alz_1000samples_csv_chunks\alz_chunk_1.csv
✅ Saved chunk 2 as CSV: alz_1000samples_csv_chunks\alz_chunk_2.csv
✅ Saved chunk 3 as CSV: alz_1000samples_csv_chunks\alz_chunk_3.csv
✅ Saved chunk 4 as CSV: alz_1000samples_csv_chunks\alz_chunk_4.csv
✅ Saved chunk 5 as CSV: alz_1000samples_csv_chunks\alz_chunk_5.csv
✅ Saved chunk 6 as CSV: alz_1000samples_csv_chunks\alz_chunk_6.csv
✅ Saved chunk 7 as CSV: alz_1000samples_csv_chunks\alz_chunk_7.csv
✅ Saved chunk 8 as CSV: alz_1000samples_csv_chunks\alz_chunk_8.csv
✅ Saved chunk 9 as CSV: alz_1000samples_csv_chunks\alz_chunk_9.csv
✅ Saved chunk 10 as CSV: alz_1000samples_csv_chunks\alz_chunk_10.csv
✅ Saved chunk 11 as CSV: alz_1000samples_csv_chunks\alz_chunk_11.csv
✅ Saved chunk 12 as CSV: alz_1000samples_csv_chunks\alz_chunk_12.csv
✅ Saved chunk 13 as CSV: alz_1000samples_csv_chunks\alz_chunk_13.csv
✅ Saved chunk 14 as CSV: alz_1000samples_csv_chunks\alz_chunk_14.csv
✅ Saved chunk 15 as CSV: alz_1000samples_csv_chunks\

In [14]:
import numpy as np

logit_val = 1.336669

# Standard logit → probability
p_standard = 1 / (1 + np.exp(-logit_val))

# Format to fixed decimal (e.g., 11 decimal places)
print("{0:.11f}".format(p_standard))
print("probabilty of logit_val is:""{0:.11f}".format(p_standard))


0.79194162600
probabilty of logit_val is:0.79194162600


In [16]:
import pandas as pd
import numpy as np
import glob
import os

# Paths — make sure to put your folder names as strings
input_dir = "alz_1000samples_csv_chunks"# folder with original chunk CSVs
output_dir = "alz_1000samples_csv_mlready"    # folder to save ML-ready probability CSVs
os.makedirs(output_dir, exist_ok=True)

# List all CSV chunk files
chunk_files = sorted(glob.glob(f"{input_dir}/*.csv"))

print(f"Found {len(chunk_files)} chunk files.")


Found 49 chunk files.


In [17]:
def process_chunk(df_chunk, var_threshold=0.001):
    """
    1. Convert logit → probability
    2. Filter CpGs by variance
    """
    # 1️⃣ Convert logit → probability
    df_prob = 1 / (1 + np.exp(-df_chunk))

    # 2️⃣ Variance filter
    variances = df_prob.var(axis=0)
    keep_cpgs = variances[variances >= var_threshold].index
    df_filtered = df_prob[keep_cpgs]

    return df_filtered


In [19]:
import pandas as pd
import numpy as np
import glob
import os

# -----------------------------
# Paths
# -----------------------------
input_dir = "alz_1000samples_csv_chunks"      # folder with original chunk CSVs
output_dir = "alz_1000samples_csv_mlready"   # folder to save ML-ready probability CSVs
os.makedirs(output_dir, exist_ok=True)

chunk_files = sorted(glob.glob(f"{input_dir}/*.csv"))
print(f"Found {len(chunk_files)} chunk files.")

# -----------------------------
# Load labels
# -----------------------------
labels_df = pd.read_csv("alzheimers_balanced.csv", index_col='sample_id')
# Create binary labels: 1 = Alzheimer, 0 = Control
y = labels_df['disease'].map(lambda x: 1 if x.lower() == 'alzheimer' else 0)

# -----------------------------
# Define processing function
# -----------------------------
def process_chunk(df_chunk, var_threshold=0.001):
    """
    1️⃣ Convert logit → probability
    2️⃣ Filter CpGs by variance
    """
    # Convert logit → probability
    df_prob = 1 / (1 + np.exp(-df_chunk))

    # Filter CpGs by variance
    variances = df_prob.var(axis=0)
    keep_cpgs = variances[variances >= var_threshold].index
    df_filtered = df_prob[keep_cpgs]

    return df_filtered

# -----------------------------
# Process all chunks
# -----------------------------
for i, f in enumerate(chunk_files):
    print(f"\nProcessing chunk {i+1}/{len(chunk_files)}: {f}")

    # Load chunk CSV
    df_chunk = pd.read_csv(f, index_col=0)

    # Re-order samples to match label file
    df_chunk = df_chunk.loc[y.index]

    # Convert + variance filter
    df_mlready = process_chunk(df_chunk, var_threshold=0.001)

    # Round for smaller file size
    df_mlready = df_mlready.round(8)

    # Save ML-ready chunk CSV
    out_file = os.path.join(output_dir, f"mlready_chunk_{i+1}.csv")
    df_mlready.to_csv(out_file)

    print(f"✅ Saved ML-ready chunk: {out_file}")


Found 49 chunk files.

Processing chunk 1/49: alz_1000samples_csv_chunks\alz_chunk_1.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_mlready\mlready_chunk_1.csv

Processing chunk 2/49: alz_1000samples_csv_chunks\alz_chunk_10.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_mlready\mlready_chunk_2.csv

Processing chunk 3/49: alz_1000samples_csv_chunks\alz_chunk_11.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_mlready\mlready_chunk_3.csv

Processing chunk 4/49: alz_1000samples_csv_chunks\alz_chunk_12.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_mlready\mlready_chunk_4.csv

Processing chunk 5/49: alz_1000samples_csv_chunks\alz_chunk_13.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_mlready\mlready_chunk_5.csv

Processing chunk 6/49: alz_1000samples_csv_chunks\alz_chunk_14.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_mlready\mlready_chunk_6.csv

Processing chunk 7/49: alz_1000samples_csv_chunks\alz_chunk_15.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_mlready\mlready_chunk_7.csv

In [4]:
import pandas as pd
import numpy as np
import glob
import os

# Paths
chunks_path = "alz_1000samples_csv_mlready/*.csv"
labels_path = "alzheimers_balanced.csv"
output_file = "alz_clean_mlready_testing.csv"

# Load labels
labels_df = pd.read_csv(labels_path, index_col=0)
y = labels_df['disease'].map(lambda x: 1 if x.lower() == 'alzheimer' else 0)

# Normalize sample IDs
y.index = y.index.astype(str).str.strip().str.lower()
print(f"Labels loaded: {y.shape[0]} samples")

all_chunks = []
for f in sorted(glob.glob(chunks_path)):
    print(f"Reading {f}...")
    df = pd.read_csv(f, index_col=0)

    # Normalize IDs
    df.index = df.index.astype(str).str.strip().str.lower()

    # Intersect with labels
    common = df.index.intersection(y.index)
    print(f"  -> {len(common)} matched samples")

    if len(common) > 0:
        df = df.loc[common]
        all_chunks.append(df)

# Combine all chunks
if len(all_chunks) == 0:
    raise ValueError("❌ No matching samples found between chunks and labels!")

X = pd.concat(all_chunks, axis=1)
print(f"Combined shape (before filtering): {X.shape}")

# Align with labels
X = X.loc[X.index.intersection(y.index)]
y = y.loc[X.index]
print(f"Final aligned shape: {X.shape}, Labels: {y.shape}")

# === STEP 3: Variance filter ===
# Use nanvar to avoid filling everything right now
variances = np.nanvar(X.values, axis=0)
var_series = pd.Series(variances, index=X.columns)

important_cpgs = var_series[var_series > 0.01].index   # tune threshold
X = X[important_cpgs]
print(f"Selected {len(important_cpgs)} CpGs")

# Add labels back
final_df = X.copy()
final_df['label'] = y

# Save
final_df.to_csv(output_file)
print(f"✅ Saved ML-ready dataset: {output_file}")


Labels loaded: 1000 samples
Reading alz_1000samples_csv_mlready\mlready_chunk_1.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_10.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_11.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_12.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_13.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_14.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_15.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_16.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_17.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_18.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_19.csv...
  -> 0 matched samples
Reading alz_1000samples_csv_mlready\mlready_chunk_2.csv...
 

ValueError: ❌ No matching samples found between chunks and labels!

In [5]:
# Show first few IDs from labels
print("\nSample IDs from labels.csv (first 5):")
print(y.index[:5].tolist())

# Show first few IDs from a chunk
test_file = sorted(glob.glob(chunks_path))[0]
df_test = pd.read_csv(test_file, index_col=0)
print(f"\nSample IDs from {os.path.basename(test_file)} (first 5):")
print(df_test.index[:5].tolist())



Sample IDs from labels.csv (first 5):
['1', '3', '4', '6', '7']

Sample IDs from mlready_chunk_1.csv (first 5):
['train10001', 'train10003', 'train10004', 'train10006', 'train10007']


In [9]:
import pandas as pd

labels_file ="alzheimers_balanced.csv"
labels_df = pd.read_csv(labels_file)

print(labels_df.head())       # see first rows
print(labels_df.columns)      # see column names


   s_no   sample_id   age gender     sample_type              disease
0     1  train10001  88.0      F  disease tissue  Alzheimer's disease
1     3  train10003  93.0      F  disease tissue  Alzheimer's disease
2     4  train10004  96.0      F  disease tissue  Alzheimer's disease
3     6  train10006  80.0      M  disease tissue  Alzheimer's disease
4     7  train10007  79.0      F  disease tissue  Alzheimer's disease
Index(['s_no', 'sample_id', 'age', 'gender', 'sample_type', 'disease'], dtype='object')


In [ ]:
import pandas as pd
import glob
import os

# Paths
labels_file ="alzheimers_balanced.csv"
chunks_path = "alz_1000samples_csv_mlready/*.csv"

# Load labels
labels_df = pd.read_csv(labels_file)
labels_df["sample_id"] = labels_df["sample_id"].astype(str)

# Map disease → binary label
labels_df["Label"] = labels_df["disease"].apply(
    lambda x: 1 if "Alzheimer" in str(x) else 0
)

# Dictionary: sample_id → binary label
labels_dict = labels_df.set_index("sample_id")["Label"].to_dict()
print(f"Labels loaded: {len(labels_dict)} samples")

# Collect matched data
all_data = []

for f in glob.glob(chunks_path):
    df = pd.read_csv(f, index_col=0)  # rows = samples, cols = CpGs
    df.index = df.index.astype(str)   # ensure IDs are strings

    # Keep only samples present in labels
    matched = df.index.intersection(labels_df["sample_id"])
    if len(matched) == 0:
        print(f"Reading {os.path.basename(f)}... -> 0 matched samples")
        continue

    df = df.loc[matched]
    # Add binary label column
    df["Label"] = df.index.map(labels_dict)

    print(f"Reading {os.path.basename(f)}... -> {len(matched)} matched samples")
    all_data.append(df)

# Combine all chunks
if all_data:
    combined = pd.concat(all_data, axis=1)

    # Deduplicate "Label" column if it appears multiple times
    if combined.columns.duplicated().any():
        combined = combined.loc[:, ~combined.columns.duplicated()]

    print("Final dataset shape:", combined.shape)

    # Save ML-ready dataset
    combined.to_csv("alz_mlready_combined.csv")
    print("✅ Saved: alz_mlready_combined.csv (binary labels ready for ML)")
else:
    print("❌ No samples matched across all chunks!")


Labels loaded: 1000 samples
Reading mlready_chunk_1.csv... -> 1000 matched samples
Reading mlready_chunk_10.csv... -> 1000 matched samples
Reading mlready_chunk_11.csv... -> 1000 matched samples
Reading mlready_chunk_12.csv... -> 1000 matched samples
Reading mlready_chunk_13.csv... -> 1000 matched samples
Reading mlready_chunk_14.csv... -> 1000 matched samples
Reading mlready_chunk_15.csv... -> 1000 matched samples
Reading mlready_chunk_16.csv... -> 1000 matched samples
Reading mlready_chunk_17.csv... -> 1000 matched samples
Reading mlready_chunk_18.csv... -> 1000 matched samples
Reading mlready_chunk_19.csv... -> 1000 matched samples
Reading mlready_chunk_2.csv... -> 1000 matched samples
Reading mlready_chunk_20.csv... -> 1000 matched samples
Reading mlready_chunk_21.csv... -> 1000 matched samples
Reading mlready_chunk_22.csv... -> 1000 matched samples
Reading mlready_chunk_23.csv... -> 1000 matched samples
Reading mlready_chunk_24.csv... -> 1000 matched samples
Reading mlready_chunk_

In [1]:
import pandas as pd
import numpy as np
import glob
import os

# -----------------------------
# Paths
# -----------------------------
input_dir = "alz_1000samples_csv_chunks"      # folder with original chunk CSVs
output_dir = "alz_1000samples_csv_prob"   # folder to save ML-ready probability CSVs
os.makedirs(output_dir, exist_ok=True)

chunk_files = sorted(glob.glob(f"{input_dir}/*.csv"))
print(f"Found {len(chunk_files)} chunk files.")

# -----------------------------
# Load labels
# -----------------------------
labels_df = pd.read_csv("alzheimers_balanced.csv", index_col='sample_id')
# Create binary labels: 1 = Alzheimer, 0 = Control
y = labels_df['disease'].map(lambda x: 1 if x.lower() == 'alzheimer' else 0)

# -----------------------------
# Define processing function
# -----------------------------
def process_chunk(df_chunk, var_threshold=0.001):
    """
    1️⃣ Convert logit → probability
    2️⃣ Filter CpGs by variance
    """
    # Convert logit → probability
    df_prob = 1 / (1 + np.exp(-df_chunk))

    # Filter CpGs by variance
    variances = df_prob.var(axis=0)
    keep_cpgs = variances[variances >= var_threshold].index
    df_filtered = df_prob[keep_cpgs]

    return df_filtered

# -----------------------------
# Process all chunks
# -----------------------------
for i, f in enumerate(chunk_files):
    print(f"\nProcessing chunk {i+1}/{len(chunk_files)}: {f}")

    # Load chunk CSV
    df_chunk = pd.read_csv(f, index_col=0)

    # Re-order samples to match label file
    df_chunk = df_chunk.loc[y.index]

    # Convert + variance filter
    df_mlready = process_chunk(df_chunk, var_threshold=0.001)

    # Round for smaller file size
    df_mlready = df_mlready.round(8)

    # Save ML-ready chunk CSV
    out_file = os.path.join(output_dir, f"mlready_chunk_{i+1}.csv")
    df_mlready.to_csv(out_file)

    print(f"✅ Saved ML-ready chunk: {out_file}")


Found 49 chunk files.

Processing chunk 1/49: alz_1000samples_csv_chunks\alz_chunk_1.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_prob\mlready_chunk_1.csv

Processing chunk 2/49: alz_1000samples_csv_chunks\alz_chunk_10.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_prob\mlready_chunk_2.csv

Processing chunk 3/49: alz_1000samples_csv_chunks\alz_chunk_11.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_prob\mlready_chunk_3.csv

Processing chunk 4/49: alz_1000samples_csv_chunks\alz_chunk_12.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_prob\mlready_chunk_4.csv

Processing chunk 5/49: alz_1000samples_csv_chunks\alz_chunk_13.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_prob\mlready_chunk_5.csv

Processing chunk 6/49: alz_1000samples_csv_chunks\alz_chunk_14.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_prob\mlready_chunk_6.csv

Processing chunk 7/49: alz_1000samples_csv_chunks\alz_chunk_15.csv
✅ Saved ML-ready chunk: alz_1000samples_csv_prob\mlready_chunk_7.csv

Processing chunk 8/

In [4]:
import os
import pandas as pd

input_dir = r"alz_1000samples_csv_prob"  # folder with your probability CSVs
output_file = "top50_cpgs_probabilities.csv"

# Ensure top50_cpgs is already defined
# Example: top50_cpgs = ['cg15265092', 'cg01053894', ...]  

first_file = True

for i, fname in enumerate(sorted(os.listdir(input_dir))):
    if not fname.endswith(".csv"):
        continue
    print(f"Processing chunk {i+1}/{len(os.listdir(input_dir))} → {fname}")
    
    df = pd.read_csv(os.path.join(input_dir, fname))
    
    # Standardize sample_id type
    df["sample_id"] = df["sample_id"].astype(str).str.strip()
    
    # Merge labels
    df = df.merge(labels_df[["sample_id", "label"]], on="sample_id", how="inner")
    
    # Select CpGs that exist in this chunk
    present_cpgs = [c for c in top50_cpgs if c in df.columns]
    df_top50 = df[["sample_id", "label"] + present_cpgs].copy()
    
    # Fill missing CpGs (from top50 but not in this chunk) with 0.5
    missing_cpgs = [c for c in top50_cpgs if c not in df.columns]
    for c in missing_cpgs:
        df_top50[c] = 0.5
    
    # Reorder columns to match top50 order
    df_top50 = df_top50[["sample_id", "label"] + top50_cpgs]
    
    # Append to output CSV
    df_top50.to_csv(output_file, mode="w" if first_file else "a",
                    header=first_file, index=False)
    first_file = False

print("✅ Done! Top 50 CpGs probability CSV saved:", output_file)


Processing chunk 2/50 → mlready_chunk_1.csv
Processing chunk 3/50 → mlready_chunk_10.csv
Processing chunk 4/50 → mlready_chunk_11.csv
Processing chunk 5/50 → mlready_chunk_12.csv
Processing chunk 6/50 → mlready_chunk_13.csv
Processing chunk 7/50 → mlready_chunk_14.csv
Processing chunk 8/50 → mlready_chunk_15.csv
Processing chunk 9/50 → mlready_chunk_16.csv
Processing chunk 10/50 → mlready_chunk_17.csv
Processing chunk 11/50 → mlready_chunk_18.csv
Processing chunk 12/50 → mlready_chunk_19.csv
Processing chunk 13/50 → mlready_chunk_2.csv
Processing chunk 14/50 → mlready_chunk_20.csv
Processing chunk 15/50 → mlready_chunk_21.csv
Processing chunk 16/50 → mlready_chunk_22.csv
Processing chunk 17/50 → mlready_chunk_23.csv
Processing chunk 18/50 → mlready_chunk_24.csv
Processing chunk 19/50 → mlready_chunk_25.csv
Processing chunk 20/50 → mlready_chunk_26.csv
Processing chunk 21/50 → mlready_chunk_27.csv
Processing chunk 22/50 → mlready_chunk_28.csv
Processing chunk 23/50 → mlready_chunk_29.cs

In [5]:
import os
import pandas as pd

input_dir = r"alz_1000samples_csv_prob"  # folder with probability CSVs
output_file = "top50_cpgs_probabilities_unique.csv"

# top 50 CpGs
# top50_cpgs = ['cg15265092', 'cg01053894', ...]  # make sure this is defined

# Dictionary to store all samples
sample_dict = {}

for i, fname in enumerate(sorted(os.listdir(input_dir))):
    if not fname.endswith(".csv"):
        continue
    print(f"Processing file {i+1}/{len(os.listdir(input_dir))} → {fname}")
    
    df = pd.read_csv(os.path.join(input_dir, fname))
    
    # Standardize sample_id
    df["sample_id"] = df["sample_id"].astype(str).str.strip()
    
    # Merge label
    df = df.merge(labels_df[["sample_id", "label"]], on="sample_id", how="inner")
    
    # Ensure all top50 CpGs are present
    for cpg in top50_cpgs:
        if cpg not in df.columns:
            df[cpg] = 0.5  # fill missing CpGs with 0.5

    # Reorder columns
    df = df[["sample_id", "label"] + top50_cpgs]
    
    # Add/update sample_dict
    for _, row in df.iterrows():
        sample_dict[row["sample_id"]] = row

# Convert dict to DataFrame (guaranteed unique sample_ids)
df_top50_unique = pd.DataFrame(sample_dict.values())

# Save to CSV
df_top50_unique.to_csv(output_file, index=False)
print(f"✅ Done! Top 50 CpGs probability CSV saved with {len(df_top50_unique)} unique samples.")


Processing file 2/50 → mlready_chunk_1.csv
Processing file 3/50 → mlready_chunk_10.csv
Processing file 4/50 → mlready_chunk_11.csv
Processing file 5/50 → mlready_chunk_12.csv
Processing file 6/50 → mlready_chunk_13.csv
Processing file 7/50 → mlready_chunk_14.csv
Processing file 8/50 → mlready_chunk_15.csv
Processing file 9/50 → mlready_chunk_16.csv
Processing file 10/50 → mlready_chunk_17.csv
Processing file 11/50 → mlready_chunk_18.csv
Processing file 12/50 → mlready_chunk_19.csv
Processing file 13/50 → mlready_chunk_2.csv
Processing file 14/50 → mlready_chunk_20.csv
Processing file 15/50 → mlready_chunk_21.csv
Processing file 16/50 → mlready_chunk_22.csv
Processing file 17/50 → mlready_chunk_23.csv
Processing file 18/50 → mlready_chunk_24.csv
Processing file 19/50 → mlready_chunk_25.csv
Processing file 20/50 → mlready_chunk_26.csv
Processing file 21/50 → mlready_chunk_27.csv
Processing file 22/50 → mlready_chunk_28.csv
Processing file 23/50 → mlready_chunk_29.csv
Processing file 24/5

In [6]:
import os
import pandas as pd

input_dir =r"alz_1000samples_csv_prob"
output_file = "top50_cpgs_probabilities_unique_new.csv"

# top 50 CpGs (make sure this is defined)
# top50_cpgs = [...]

sample_dict = {}

for i, fname in enumerate(sorted(os.listdir(input_dir))):
    if not fname.endswith(".csv"):
        continue
    print(f"Processing file {i+1}/{len(os.listdir(input_dir))} → {fname}")
    
    df = pd.read_csv(os.path.join(input_dir, fname))
    
    # Strip and lowercase column names to standardize
    df.columns = df.columns.str.strip().str.lower()
    
    df["sample_id"] = df["sample_id"].astype(str).str.strip()
    
    # Merge label
    df = df.merge(labels_df[["sample_id", "label"]], on="sample_id", how="inner")
    
    # Standardize top50 CpGs to lowercase too
    top50_lower = [c.lower() for c in top50_cpgs]
    
    # For missing CpGs only
    for cpg, cpg_lower in zip(top50_cpgs, top50_lower):
        if cpg_lower not in df.columns:
            df[cpg_lower] = 0.5  # fill missing CpG with 0.5

    # Reorder columns
    df = df[["sample_id", "label"] + top50_lower]
    
    # Add/update sample_dict
    for _, row in df.iterrows():
        sample_dict[row["sample_id"]] = row

# Convert dict to DataFrame
df_top50_unique = pd.DataFrame(sample_dict.values())

# Rename columns back to original names
df_top50_unique.columns = ["sample_id", "label"] + top50_cpgs

# Save to CSV
df_top50_unique.to_csv(output_file, index=False)
print(f"✅ Done! Top 50 CpGs probability CSV saved with {len(df_top50_unique)} unique samples.")


Processing file 2/50 → mlready_chunk_1.csv
Processing file 3/50 → mlready_chunk_10.csv
Processing file 4/50 → mlready_chunk_11.csv
Processing file 5/50 → mlready_chunk_12.csv
Processing file 6/50 → mlready_chunk_13.csv
Processing file 7/50 → mlready_chunk_14.csv
Processing file 8/50 → mlready_chunk_15.csv
Processing file 9/50 → mlready_chunk_16.csv
Processing file 10/50 → mlready_chunk_17.csv
Processing file 11/50 → mlready_chunk_18.csv
Processing file 12/50 → mlready_chunk_19.csv
Processing file 13/50 → mlready_chunk_2.csv
Processing file 14/50 → mlready_chunk_20.csv
Processing file 15/50 → mlready_chunk_21.csv
Processing file 16/50 → mlready_chunk_22.csv
Processing file 17/50 → mlready_chunk_23.csv
Processing file 18/50 → mlready_chunk_24.csv
Processing file 19/50 → mlready_chunk_25.csv
Processing file 20/50 → mlready_chunk_26.csv
Processing file 21/50 → mlready_chunk_27.csv
Processing file 22/50 → mlready_chunk_28.csv
Processing file 23/50 → mlready_chunk_29.csv
Processing file 24/5

In [11]:
import os
import pandas as pd

# -----------------------------
# Paths & parameters
# -----------------------------
prob_folder = r"alz_1000samples_csv_prob" # folder with your 49 probability CSVs
labels_file = r"C:\Users\rakes\Documents\neet 2025\capstone\alzheimers_balanced.csv"     # CSV with sample_id + label
output_file =  "top50_cpgs_probabilities_unique_final.csv" # final output CSV
top50_cpgs = [                              
    'cg15265092','cg01053894','cg25103043','cg01908874','cg10530613',
    'cg16909341','cg06242416','ch.12.2933188F','cg07565150','cg14224760',
    'cg07589531','cg15405121','cg11508528','cg03312856','cg04918002',
    'cg16908556','cg22052758','cg03715182','cg07601463','cg06746362',
    'cg11461302','cg14569423','cg12320198','cg26852404','cg02129146',
    'cg03552317','cg24030996','cg08693082','cg01445325','cg04247746',
    'cg11882601','cg14041338','cg26768765','cg12114985','cg24791283',
    'cg03039837','cg07943849','cg09250423','cg14555759','cg10617091',
    'cg17183770','cg08715988','cg19583880','cg12482564','cg20250700',
    'cg15451585','cg03756034','cg05084610','cg02755555','cg00562011'
]

# -----------------------------
# Load labels and encode
# -----------------------------
labels_df = pd.read_csv(labels_file)
labels_df["sample_id"] = labels_df["sample_id"].astype(str).str.strip()

# Convert disease → numeric label (1 = Alzheimer's, 0 = control)
labels_df["label"] = labels_df["disease"].map({"Alzheimer's disease": 1, "control": 0})

# -----------------------------
# Process files
# -----------------------------
first_file = True
csv_files = sorted([f for f in os.listdir(prob_folder) if f.endswith(".csv")])

for idx, f in enumerate(csv_files, 1):
    file_path = os.path.join(prob_folder, f)
    print(f"Processing file {idx}/{len(csv_files)} → {f}")

    df = pd.read_csv(file_path)
    df["sample_id"] = df["sample_id"].astype(str).str.strip()

    # Merge numeric label
    df = df.merge(labels_df[["sample_id", "label"]], on="sample_id", how="inner")

    # Keep only top50 CpGs that exist in this file
    existing_cpgs = [c for c in top50_cpgs if c in df.columns]
    df_top50 = df[["sample_id", "label"] + existing_cpgs].copy()

    # Fill missing CpGs with 0.5
    missing_cpgs = [c for c in top50_cpgs if c not in existing_cpgs]
    for c in missing_cpgs:
        df_top50.loc[:, c] = 0.5

    # Save/append to CSV
    df_top50.to_csv(output_file, mode="w" if first_file else "a",
                    header=first_file, index=False)
    first_file = False

print(f"✅ Done! Top 50 CpG probabilities saved to: {output_file}")


Processing file 1/49 → mlready_chunk_1.csv
Processing file 2/49 → mlready_chunk_10.csv
Processing file 3/49 → mlready_chunk_11.csv
Processing file 4/49 → mlready_chunk_12.csv
Processing file 5/49 → mlready_chunk_13.csv
Processing file 6/49 → mlready_chunk_14.csv
Processing file 7/49 → mlready_chunk_15.csv
Processing file 8/49 → mlready_chunk_16.csv
Processing file 9/49 → mlready_chunk_17.csv
Processing file 10/49 → mlready_chunk_18.csv
Processing file 11/49 → mlready_chunk_19.csv
Processing file 12/49 → mlready_chunk_2.csv
Processing file 13/49 → mlready_chunk_20.csv
Processing file 14/49 → mlready_chunk_21.csv
Processing file 15/49 → mlready_chunk_22.csv
Processing file 16/49 → mlready_chunk_23.csv
Processing file 17/49 → mlready_chunk_24.csv
Processing file 18/49 → mlready_chunk_25.csv
Processing file 19/49 → mlready_chunk_26.csv
Processing file 20/49 → mlready_chunk_27.csv
Processing file 21/49 → mlready_chunk_28.csv
Processing file 22/49 → mlready_chunk_29.csv
Processing file 23/49

In [15]:
import pandas as pd
import glob
import os

# -----------------------------
# 1️⃣ Folder paths
# -----------------------------
input_folder = "alz_1000samples_csv_prob"   # folder containing mlready_chunk_*.csv
output_file = "alz_merged_cleaned_with_new_probs.csv"         # final merged & cleaned file

# -----------------------------
# 2️⃣ Find all chunk files
# -----------------------------
chunk_files = sorted(glob.glob(os.path.join(input_folder, "mlready_chunk_*.csv")))
print(f"✅ Found {len(chunk_files)} chunk files")

if not chunk_files:
    raise FileNotFoundError(f"No chunk files found in: {input_folder}")

# -----------------------------
# 3️⃣ Stream through each file → Clean → Append
# -----------------------------
first = True
for i, f in enumerate(chunk_files, start=1):
    print(f"📄 Processing chunk {i}/{len(chunk_files)} → {os.path.basename(f)}")
    
    # Read chunk
    df = pd.read_csv(f)
    
    # Fill NaNs with 0.5 (neutral methylation)
    df = df.fillna(0.5)
    
    # Round all CpG probability values to 6 decimal places
    df = df.round(6)
    
    # Write to output file
    if first:
        df.to_csv(output_file, mode='w', index=False)  # Write header for the first chunk
        first = False
    else:
        df.to_csv(output_file, mode='a', index=False, header=False)  # Append without header

print(f"\n🎉 All chunks merged & cleaned successfully!")
print(f"📁 Output saved as: {output_file}")


✅ Found 49 chunk files
📄 Processing chunk 1/49 → mlready_chunk_1.csv
📄 Processing chunk 2/49 → mlready_chunk_10.csv
📄 Processing chunk 3/49 → mlready_chunk_11.csv
📄 Processing chunk 4/49 → mlready_chunk_12.csv
📄 Processing chunk 5/49 → mlready_chunk_13.csv
📄 Processing chunk 6/49 → mlready_chunk_14.csv
📄 Processing chunk 7/49 → mlready_chunk_15.csv
📄 Processing chunk 8/49 → mlready_chunk_16.csv
📄 Processing chunk 9/49 → mlready_chunk_17.csv
📄 Processing chunk 10/49 → mlready_chunk_18.csv
📄 Processing chunk 11/49 → mlready_chunk_19.csv
📄 Processing chunk 12/49 → mlready_chunk_2.csv
📄 Processing chunk 13/49 → mlready_chunk_20.csv
📄 Processing chunk 14/49 → mlready_chunk_21.csv
📄 Processing chunk 15/49 → mlready_chunk_22.csv
📄 Processing chunk 16/49 → mlready_chunk_23.csv
📄 Processing chunk 17/49 → mlready_chunk_24.csv
📄 Processing chunk 18/49 → mlready_chunk_25.csv
📄 Processing chunk 19/49 → mlready_chunk_26.csv
📄 Processing chunk 20/49 → mlready_chunk_27.csv
📄 Processing chunk 21/49 → m

In [1]:
import pandas as pd
import glob
import os

# -----------------------------
# 1. Folder paths
# -----------------------------
input_folder = "alz_1000samples_csv_mlready"   # folder with 49 chunk files
output_parquet = "alz_merged_cleaned_prob_latest.parquet"  # final cleaned dataset

# -----------------------------
# 2. Get all chunk files
# -----------------------------
chunk_files = sorted(glob.glob(os.path.join(input_folder, "mlready_chunk_*.csv")))
print(f"✅ Found {len(chunk_files)} chunk files")

if not chunk_files:
    raise FileNotFoundError(f"No chunk files found in folder: {input_folder}")

# -----------------------------
# 3. Clean and append each chunk
# -----------------------------
for i, f in enumerate(chunk_files, start=1):
    print(f"📄 Cleaning chunk {i}/{len(chunk_files)} → {os.path.basename(f)}")
    
    # Load chunk
    df = pd.read_csv(f)
    
    # Fill NaN with 0.5 (neutral methylation)
    df = df.fillna(0.5)
    
    # Round CpG values to 6 decimal places
    df = df.round(6)
    
    # Append to parquet file incrementally
    if i == 1:
        df.to_parquet(output_parquet, engine='pyarrow', index=False)
    else:
        # For multiple parquet files: write separately then merge later
        df.to_parquet(f"temp_chunk_{i}.parquet", engine='pyarrow', index=False)

print("\n✅ All chunks cleaned and saved as parquet parts.")
print(f"📝 First cleaned chunk saved as: {output_parquet}")


✅ Found 49 chunk files
📄 Cleaning chunk 1/49 → mlready_chunk_1.csv
📄 Cleaning chunk 2/49 → mlready_chunk_10.csv
📄 Cleaning chunk 3/49 → mlready_chunk_11.csv
📄 Cleaning chunk 4/49 → mlready_chunk_12.csv
📄 Cleaning chunk 5/49 → mlready_chunk_13.csv
📄 Cleaning chunk 6/49 → mlready_chunk_14.csv
📄 Cleaning chunk 7/49 → mlready_chunk_15.csv
📄 Cleaning chunk 8/49 → mlready_chunk_16.csv
📄 Cleaning chunk 9/49 → mlready_chunk_17.csv
📄 Cleaning chunk 10/49 → mlready_chunk_18.csv
📄 Cleaning chunk 11/49 → mlready_chunk_19.csv
📄 Cleaning chunk 12/49 → mlready_chunk_2.csv
📄 Cleaning chunk 13/49 → mlready_chunk_20.csv
📄 Cleaning chunk 14/49 → mlready_chunk_21.csv
📄 Cleaning chunk 15/49 → mlready_chunk_22.csv
📄 Cleaning chunk 16/49 → mlready_chunk_23.csv
📄 Cleaning chunk 17/49 → mlready_chunk_24.csv
📄 Cleaning chunk 18/49 → mlready_chunk_25.csv
📄 Cleaning chunk 19/49 → mlready_chunk_26.csv
📄 Cleaning chunk 20/49 → mlready_chunk_27.csv
📄 Cleaning chunk 21/49 → mlready_chunk_28.csv
📄 Cleaning chunk 22/49

In [6]:
from IPython.display import FileLink
FileLink("alz_merged_cleaned_prob_latest.parquet" )


C:\Users\rakes\alz_merged_cleaned_prob_latest.parquet

In [7]:
 "alz_merged_cleaned_with_new_probs.csv"

'alz_merged_cleaned_with_new_probs.csv'

In [8]:
from IPython.display import FileLink
FileLink( "alz_merged_cleaned_with_new_probs.csv" )

C:\Users\rakes\alz_merged_cleaned_with_new_probs.csv

In [1]:
import shutil
shutil.make_archive("alz_prob_chunks", 'zip', "alz_1000samples_csv_prob")


'C:\\Users\\rakes\\alz_prob_chunks.zip'

In [3]:
import shutil

# Folder that contains your 49 chunk CSV files
folder_to_zip = "alz_1000samples_csv_prob"

# Name for the output zip file
output_zip = "alz_prob_chunks"

# Create zip
shutil.make_archive(output_zip, 'zip', folder_to_zip)
print("✅ Folder zipped successfully →", output_zip + ".zip")


✅ Folder zipped successfully → alz_prob_chunks.zip


In [4]:
from IPython.display import FileLink
FileLink('alz_prob_chunks.zip')


C:\Users\rakes\alz_prob_chunks.zip